# Fertilizer Recommendation Model
### AgriSense — IoT & AI Based Smart Crop and Fertilizer Recommendation System
**Team:** Kushal | Akarshan Poudel | Sachin Kandel | Sushovan Bikram Shahi

---
**Input Sources:**
| Feature | Source |
|---|---|
| N, P, K | Manually entered in UI |
| Soil_pH, Soil_Moisture, Temperature, Humidity | Sensors (ESP32) |
| Rainfall | Manually entered in UI |
| Soil_Type, Crop_Type | UI Dropdown |

**Output:** Recommended fertilizer name


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.figsize'] = (10, 5)
print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Dataset

In [2]:
df = pd.read_csv("fertilizer_recommendation (1).csv")
print("Shape:", df.shape)
df.head(10)

Shape: (2100, 11)


,Soil_Type,Soil_pH,Soil_Moisture,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall,Crop_Type,Recommended_Fertilizer
0,Loamy,8.35,53.10,153,25,58,39.55,44.53,2370.25,Cotton,DAP
1,Silt,5.86,36.36,108,47,107,38.43,73.34,709.32,Cotton,MOP
2,Sandy,8.00,10.54,115,78,79,15.03,40.78,696.76,Cotton,Zinc Sulphate
3,Sandy,7.49,29.64,118,67,28,26.04,77.27,1947.20,Cotton,MOP
4,Loamy,8.23,31.37,147,84,66,10.12,40.79,1733.06,Cotton,Zinc Sulphate
5,Loamy,5.68,52.49,153,81,29,18.35,89.20,2648.49,Cotton,MOP
6,Loamy,4.61,17.39,25,18,67,30.57,56.47,663.62,Cotton,Urea
7,Loamy,8.23,24.83,20,23,90,19.71,67.87,1104.87,Cotton,Urea
8,Clay,5.90,21.81,122,56,112,26.98,75.45,2748.92,Cotton,Compost
9,Loamy,8.20,25.82,127,71,102,32.59,81.46,428.69,Cotton,Zinc Sulphate


## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Info

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2100 entries, 0 to 2099
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Soil_Type               2100 non-null   str    
 1   Soil_pH                 2100 non-null   float64
 2   Soil_Moisture           2100 non-null   float64
 3   Nitrogen_Level          2100 non-null   int64  
 4   Phosphorus_Level        2100 non-null   int64  
 5   Potassium_Level         2100 non-null   int64  
 6   Temperature             2100 non-null   float64
 7   Humidity                2100 non-null   float64
 8   Rainfall                2100 non-null   float64
 9   Crop_Type               2100 non-null   str    
 10  Recommended_Fertilizer  2100 non-null   str    
dtypes: float64(5), int64(3), str(3)
memory usage: 180.6 KB


In [4]:
df.describe()

,Soil_pH,Soil_Moisture,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall
count,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000
mean,6.453905,35.304010,88.801905,49.512857,64.173810,25.162767,59.980762,1572.069471
std,1.164190,14.450607,40.482874,23.059554,31.946325,8.579597,17.594396,801.342127
min,4.500000,10.020000,20.000000,10.000000,10.000000,10.000000,30.010000,201.770000
25%,5.420000,22.930000,53.000000,30.000000,37.000000,17.687500,44.782500,868.030000
50%,6.430000,35.720000,89.000000,49.000000,63.000000,25.285000,60.280000,1579.460000
75%,7.472500,47.640000,124.250000,69.000000,92.000000,32.762500,75.550000,2261.722500
max,8.500000,60.000000,159.000000,89.000000,119.000000,39.990000,89.990000,2998.860000


### 3.2 Missing Values & Duplicates

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

### 3.3 Class Distribution — Fertilizer, Crop Type, Soil Type

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in zip(axes,
    ['Recommended_Fertilizer', 'Crop_Type', 'Soil_Type'],
    ['Fertilizer Distribution', 'Crop Type Distribution', 'Soil Type Distribution']):
    counts = df[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, palette="Set2", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 3.4 Feature Distributions — Histograms

In [ ]:
num_cols = ['Soil_pH', 'Soil_Moisture', 'Nitrogen_Level', 'Phosphorus_Level',
             'Potassium_Level', 'Temperature', 'Humidity', 'Rainfall']

df[num_cols].hist(bins=30, figsize=(14, 8), color='mediumseagreen', edgecolor='black')
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

### 3.5 Boxplots — Checking for Outliers

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(num_cols):
    ax = axes[i // 4][i % 4]
    sns.boxplot(y=df[col], ax=ax, color='lightblue')
    ax.set_title(col)

plt.suptitle("Boxplots — Outlier Detection", fontsize=14)
plt.tight_layout()
plt.show()

### 3.6 Correlation Heatmap

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

### 3.7 Fertilizer Recommendation by Crop Type

In [ ]:
ct = pd.crosstab(df['Crop_Type'], df['Recommended_Fertilizer'])
ct.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='tab10')
plt.title("Fertilizer Distribution by Crop Type")
plt.xlabel("Crop Type")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.legend(title="Fertilizer", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 3.8 Fertilizer Distribution by Soil Type

In [ ]:
ct2 = pd.crosstab(df['Soil_Type'], df['Recommended_Fertilizer'])
ct2.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='Set3')
plt.title("Fertilizer Distribution by Soil Type")
plt.xlabel("Soil Type")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.legend(title="Fertilizer", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 3.9 Average NPK per Fertilizer

In [ ]:
fert_mean = df.groupby('Recommended_Fertilizer')[['Nitrogen_Level','Phosphorus_Level','Potassium_Level']].mean()
fert_mean.plot(kind='bar', figsize=(10, 5), colormap='viridis')
plt.title("Average N, P, K Level per Fertilizer")
plt.xlabel("Fertilizer")
plt.ylabel("Level")
plt.xticks(rotation=30)
plt.legend(title="Nutrient")
plt.tight_layout()
plt.show()

## 4. Preprocessing

### 4.1 Encode Categorical Columns

In [ ]:
le_soil      = LabelEncoder()
le_crop_type = LabelEncoder()
le_fert      = LabelEncoder()

df['Soil_Type_enc']  = le_soil.fit_transform(df['Soil_Type'])
df['Crop_Type_enc']  = le_crop_type.fit_transform(df['Crop_Type'])
df['Fertilizer_enc'] = le_fert.fit_transform(df['Recommended_Fertilizer'])

print("Soil types  :", le_soil.classes_.tolist())
print("Crop types  :", le_crop_type.classes_.tolist())
print("Fertilizers :", le_fert.classes_.tolist())

### 4.2 Define Features (X) and Target (y)

In [ ]:
feature_cols = ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level',
                'Soil_pH', 'Soil_Moisture', 'Temperature', 'Humidity',
                'Rainfall', 'Soil_Type_enc', 'Crop_Type_enc']

X = df[feature_cols]
y = df['Fertilizer_enc']

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

### 4.3 Train / Test Split — 80% train, 20% test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape[0])
print("Test  size:", X_test.shape[0])

## 5. Model Training — Random Forest

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline.fit(X_train, y_train)
print("Model trained successfully.")

## 6. Evaluation

### 6.1 Accuracy

In [ ]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy  : {acc * 100:.2f}%")

cv = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"5-Fold CV Mean : {cv.mean()*100:.2f}%")
print(f"5-Fold CV Std  : {cv.std()*100:.2f}%")

### 6.2 Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=le_fert.classes_))

### 6.3 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le_fert.classes_)
disp.plot(ax=ax, xticks_rotation=30, colorbar=True, cmap='Oranges')
plt.title("Confusion Matrix — Fertilizer Recommendation Model")
plt.tight_layout()
plt.show()

### 6.4 Feature Importance

In [ ]:
importances = pipeline.named_steps['rf'].feature_importances_
imp_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
imp_df = imp_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=imp_df, x='Importance', y='Feature', palette='magma')
plt.title("Feature Importance — Fertilizer Recommendation Model")
plt.tight_layout()
plt.show()

print(imp_df.to_string(index=False))

## 7. Save Model & Encoders

In [ ]:
import os
save_dir = "models"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(pipeline,      os.path.join(save_dir, "fertilizer_model.pkl"))
joblib.dump(le_soil,       os.path.join(save_dir, "soil_type_encoder.pkl"))
joblib.dump(le_crop_type,  os.path.join(save_dir, "crop_type_encoder.pkl"))
joblib.dump(le_fert,       os.path.join(save_dir, "fertilizer_label_encoder.pkl"))

print("Saved:")
print(f"  {save_dir}/fertilizer_model.pkl")
print(f"  {save_dir}/soil_type_encoder.pkl")
print(f"  {save_dir}/crop_type_encoder.pkl")
print(f"  {save_dir}/fertilizer_label_encoder.pkl")

## 8. Quick Prediction Test

In [ ]:
sample = {
    "Nitrogen_Level":   90,      # manual UI
    "Phosphorus_Level": 42,      # manual UI
    "Potassium_Level":  43,      # manual UI
    "Soil_pH":          6.5,     # pH sensor
    "Soil_Moisture":    35.0,    # soil moisture sensor
    "Temperature":      28.0,    # DHT22 sensor
    "Humidity":         70.0,    # DHT22 sensor
    "Rainfall":         800.0,   # manual UI
    "Soil_Type":        "Loamy", # UI dropdown
    "Crop_Type":        "Rice"   # UI dropdown
}

soil_enc = le_soil.transform([sample["Soil_Type"]])[0]
crop_enc = le_crop_type.transform([sample["Crop_Type"]])[0]

X_sample = [[
    sample["Nitrogen_Level"], sample["Phosphorus_Level"], sample["Potassium_Level"],
    sample["Soil_pH"], sample["Soil_Moisture"], sample["Temperature"],
    sample["Humidity"], sample["Rainfall"], soil_enc, crop_enc
]]

pred_enc   = pipeline.predict(X_sample)[0]
proba      = pipeline.predict_proba(X_sample)[0]
fert_name  = le_fert.inverse_transform([pred_enc])[0]
confidence = round(proba[pred_enc] * 100, 2)

print(f"Recommended Fertilizer : {fert_name}")
print(f"Confidence             : {confidence}%")